# Israeli 26th Knesset results map

This is the 2026 version of the repository's interactive election map. It joins each ballot box to its polling-place coordinates and builds polling-place Voronoi regions clipped to the corresponding locality boundary. Hover for a quick summary; click a region for its party chart.

### Map data

The boundary layer follows the pipeline used by [RavBogard/israelielection](https://github.com/RavBogard/israelielection/blob/main/scripts/vote-map/build.mts): Ministry of Transport 2026 locality polygons, supplemented by dissolved CBS 2008 statistical areas for West Bank localities. Full source URLs and caveats are in [2026/MAP_SOURCES.md](2026/MAP_SOURCES.md).

In [1]:
%matplotlib widget
import json
from io import StringIO
from math import cos, pi, sin
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from shapely import set_precision
from shapely.geometry import MultiPoint, Point, mapping, shape
from shapely.ops import voronoi_diagram
from ipyleaflet import (
    FullScreenControl, GeoJSON, Map, ScaleControl,
    basemaps,
)
from ipywidgets import HBox, HTML, Label, Layout, VBox

from behirot_2026 import (
    download_election_file, download_polling_places, topojson_to_geojson,
)

## Refresh the official results

This downloads and caches the official ballot-box export and polling-place table under 2026/. Results and locations are joined by the `ברזל` identifier.

In [2]:
ballot_results = download_election_file("expb.csv")
polling_places = download_polling_places()

def clean_export(frame):
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed:")].copy()
    numeric_columns = frame.columns.difference(["שם ישוב"])
    frame[numeric_columns] = frame[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0)
    return frame

ballot_results = clean_export(ballot_results)
ballot_results["ברזל"] = ballot_results["ברזל"].astype("Int64").astype("string")

party_column_index = ballot_results.columns.get_loc("קלפי אחרת") + 1
party_columns = list(ballot_results.columns[party_column_index:])
location_columns = [
    "ברזל", "סמל קלפי", "כתובת קלפי", "מקום קלפי",
    "קו אורך", "קו רוחב",
]
ballot_results = ballot_results.merge(
    polling_places[location_columns],
    on="ברזל", how="left", validate="one_to_one",
    suffixes=("", "_מיקום"),
)
located_results = ballot_results.dropna(subset=["קו אורך", "קו רוחב"]).copy()
if len(located_results) != len(ballot_results):
    missing = len(ballot_results) - len(located_results)
    print(f"{missing} result rows have no polling-place coordinates")

# Polling places missing from the results export have not been counted yet.
# Add them with zero votes so they get their own (uncounted) map cells.
uncounted_places = polling_places[
    ~polling_places["ברזל"].astype("string").isin(ballot_results["ברזל"])
].rename(columns={"סמל ישוב בחירות": "סמל ישוב", "שם ישוב בחירות": "שם ישוב"})
uncounted_places = uncounted_places.assign(
    **{"ברזל": uncounted_places["ברזל"].astype("string"),
       "קלפי": pd.to_numeric(uncounted_places["סמל קלפי"], errors="coerce")}
)[["סמל ישוב", "שם ישוב", "קלפי", *location_columns]]
count_columns = ["מצביעים", "פסולים", "כשרים", *party_columns]
uncounted_places[count_columns] = 0
located_results = pd.concat([located_results, uncounted_places], ignore_index=True)
print(f"{len(uncounted_places)} polling places not counted yet")

1862 polling places not counted yet


## Build polling-place regions

In [3]:
locality_boundaries = topojson_to_geojson("2026/boundaries.topo.json")
boundary_by_code = {}
for feature in locality_boundaries["features"]:
    boundary = shape(feature["geometry"])
    if not boundary.is_valid:
        boundary = boundary.buffer(0)
    boundary_by_code[int(feature["properties"]["code"])] = boundary

def display_ballot_number(value):
    value = str(value)
    return value[:-2] if value.endswith(".0") else value

def ballot_properties(row):
    votes = row[party_columns].astype(int)
    dominant_party = votes.idxmax() if votes.sum() else ""
    address = " ".join(
        value for value in [row["כתובת קלפי"], row["מקום קלפי"]]
        if isinstance(value, str) and value.strip()
    )
    return {
        "name": row["שם ישוב"],
        "address": address,
        "number": display_ballot_number(row["קלפי"]),
        "iron_id": str(row["ברזל"]),
        "voted": int(row["מצביעים"]),
        "invalid": int(row["פסולים"]),
        "valid": int(row["כשרים"]),
        "votes": {party: int(votes[party]) for party in party_columns},
        "dominant_party": dominant_party,
        "counted": bool(row["מצביעים"]),
    }

def separated_ballot_points(rows, distance=0.00002):
    ballot_points = []
    for (longitude, latitude), shared_rows in rows.groupby(
        ["קו אורך", "קו רוחב"], sort=False
    ):
        shared_rows = shared_rows.sort_values("קלפי")
        count = len(shared_rows)
        for index, (_, row) in enumerate(shared_rows.iterrows()):
            if count == 1:
                point = Point(float(longitude), float(latitude))
            else:
                angle = 2 * pi * index / count
                point = Point(
                    float(longitude) + distance * cos(angle),
                    float(latitude) + distance * sin(angle),
                )
            ballot_points.append((point, row))
    return ballot_points

polling_place_features = []
for locality_code, locality_rows in located_results.groupby("סמל ישוב"):
    boundary = boundary_by_code.get(int(locality_code))
    if boundary is None:
        continue

    ballots = separated_ballot_points(locality_rows)
    points = [point for point, _ in ballots]
    if len(points) == 1:
        regions = [boundary]
    else:
        diagram = voronoi_diagram(MultiPoint(points), envelope=boundary.envelope)
        cells = list(diagram.geoms)
        regions = [
            min(cells, key=lambda cell: cell.distance(point)).intersection(boundary)
            for point in points
        ]

    for (_, row), region in zip(ballots, regions):
        if region.is_empty:
            continue
        polling_place_features.append({
            "type": "Feature",
            "geometry": mapping(region),
            "properties": ballot_properties(row),
        })

polling_place_regions = {
    "type": "FeatureCollection",
    "features": polling_place_features,
}


## Interactive map

In [4]:
plt.ioff()
fig, party_ax = plt.subplots(1, 1)
plt.ion()

def plot_party_votes_df(party_votes, ax):
    ax.cla()
    plot_data = party_votes.sort_values().copy()
    total = plot_data.sum()
    if not total:
        return
    plot_data = plot_data * 100 / total
    plot_data.plot(kind='barh', ax=ax)
    ax.set_ylim([max(len(plot_data)-10, 0), len(plot_data)])
    ax.set_xlim([0,100])
    ax.xaxis.set_major_formatter(PercentFormatter())

title_widget = HTML()

def set_format(fmt):
    def set_in_fmt(widget, variable):
        widget.value = fmt % variable
    return set_in_fmt

set_title = set_format("<h1>%s</h1>")
set_voted = set_format('מצביעים: %s')
set_invalid = set_format('פסולים: %s, %.2f%%')
set_valid = set_format('כשרים: %s, %.2f%%')
set_dominant_party = set_format('מפלגה דומיננטית: %s')
set_dominant_party_votes = set_format('הצבעות למלגה הדומיננטית: %s, %.2f%%')

voted_label = Label(style={"font_size": "23px"})
invalid_label = Label(style={"font_size": "23px"})
valid_label = Label(style={"font_size": "23px"})
dominant_party_label = Label(style={"font_size": "23px", "font_weight": "bold"})
dominant_party_votes_label = Label(style={"font_size": "23px"})

set_voted(voted_label, '')
set_invalid(invalid_label, ('', 0))
set_valid(valid_label, ('', 0))
set_title(title_widget, "Hover to display quick summary")
set_dominant_party(dominant_party_label, '')
set_dominant_party_votes(dominant_party_votes_label, ('', 0))

def get_dominant_party(party):
    total = party.sum()
    if not total:
        return '', 0, 0
    name = party.idxmax()
    votes = party.loc[name]
    return name, votes, votes/total

def show_ballot(feature, plot=False):
    properties = feature["properties"]
    party = pd.Series(properties["votes"])
    title = f'{properties["name"]} — קלפי {properties["number"]}'
    if not properties["counted"]:
        title += ' (טרם נספרה)'
    if properties["address"]:
        title += f'<br><small>{properties["address"]}</small>'
    set_title(title_widget, title)
    voted = properties["voted"]
    invalid = properties["invalid"]
    valid = properties["valid"]
    set_voted(voted_label, voted)
    set_valid(valid_label, (valid, valid*100/voted if voted else 0))
    set_invalid(invalid_label, (invalid, invalid*100/voted if voted else 0))
    name, votes, percent = get_dominant_party(party)
    set_dominant_party(dominant_party_label, name)
    set_dominant_party_votes(dominant_party_votes_label, (votes, percent*100))
    if plot:
        plot_party_votes_df(party, party_ax)
        party_ax.figure.suptitle(f'{properties["name"]} — קלפי {properties["number"]}')
        party_ax.figure.canvas.draw_idle()

labels = VBox([title_widget, dominant_party_label, voted_label, valid_label, invalid_label, dominant_party_votes_label])

In [5]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(31.5, 34.9),
    zoom=7,
    min_zoom=6,
    max_zoom=15,
    scroll_wheel_zoom=True,
    prefer_canvas=True,
    world_copy_jump=False,
    layout=Layout(height="760px", width="60%"),
)

party_colors = {
    "מחל": "#245b94",  # Likud
    "ב": "#00a6c8",    # Otzma Yehudit
    "ט": "#4dd0e1",    # Religious Zionism–Zehut
    "ני": "#b29bda",   # Noam
    "ך": "#597fbd",    # Amcha Yisrael
    "שס": "#3f444a",   # Shas
    "ג": "#111111",    # United Torah Judaism
    "רק": "#ce542d",   # Beyachad
    "דרך": "#ed9859",  # Yashar
    "אמת": "#a72f65", # The Democrats
    "ל": "#9e3826",    # Yisrael Beiteinu
    "כן": "#db7898",   # Blue and White
    "די": "#b48727",   # Reservists–Economic
    "ודם": "#267b44",  # Joint List
    "עם": "#80b667",   # Ra'am
}
party_fallback_color = "#8c939b"
uncounted_color = "#ffd60a"  # polling places not counted yet
party_names = {
    "מחל": "הליכוד",
    "ב": "עוצמה יהודית",
    "ט": "הציונות הדתית–זהות",
    "ני": "נעם",
    "ך": "עמך ישראל",
    "שס": "ש״ס",
    "ג": "יהדות התורה",
    "רק": "ביחד",
    "דרך": "ישר",
    "אמת": "הדמוקרטים",
    "ל": "ישראל ביתנו",
    "כן": "כחול לבן",
    "די": "המילואימניקים–הכלכלית",
    "ודם": "הרשימה המשותפת",
    "עם": "רע״ם",
}

def dominant_party_color(feature):
    if not feature["properties"]["counted"]:
        return {'color': 'black', 'fillColor': uncounted_color}
    return {
        'color': 'black',
        'fillColor': party_colors.get(
            feature["properties"].get("dominant_party"),
            party_fallback_color,
        ),
    }

geo_json = GeoJSON(
    data=polling_place_regions,
    style={
        'opacity': 1, 'dashArray': '9', 'fillOpacity': 0.5, 'weight': 1
    },
    hover_style={
        'color': 'white', 'dashArray': '0', 'fillOpacity': 0.9
    },
    style_callback=dominant_party_color,
)
geo_json.on_hover(lambda feature, **kwargs: show_ballot(feature))
geo_json.on_click(lambda feature, **kwargs: show_ballot(feature, plot=True))
m.add(geo_json)

city_lines = GeoJSON(
    data=locality_boundaries,
    style={
        'color': '#000000', 'weight': 2, 'opacity': 1,
        'fill': False, 'interactive': False,
    },
)
m.add(city_lines)

panel = VBox(
    [labels, fig.canvas],
    layout=Layout(width="40%", max_height="760px"),
)
m.add(ScaleControl(position="bottomleft", imperial=False))
m.add(FullScreenControl(position="topleft"))
map_with_panel = HBox(
    [m, panel],
    layout=Layout(width="90vw"),
);



In [6]:
map_with_panel

In [7]:
total_party_votes = ballot_results[party_columns].sum().astype(int)
total_party_votes = total_party_votes[total_party_votes > 0].sort_values()
total_ballots_cast = int(ballot_results["מצביעים"].sum())
total_valid_votes = int(ballot_results["כשרים"].sum())
party_percent = total_party_votes * 100 / total_valid_votes

surplus_agreements = [
    ("ל", "רק"),     # Yisrael Beiteinu + Beyachad
    ("אמת", "דרך"),  # The Democrats + Yashar
    ("מחל", "ט"),    # Likud + Religious Zionism
    ("עם", "ודם"),   # Ra'am + Joint List
    ("שס", "ג"),     # Shas + United Torah Judaism
]

def dhondt(votes, seats):
    allocation = pd.Series(0, index=votes.index)
    for _ in range(seats):
        allocation[(votes / (allocation + 1)).idxmax()] += 1
    return allocation

def estimate_seats(votes, total_valid, seats=120, threshold=0.0325, agreements=()):
    # Bader-Ofer among lists over the threshold. A surplus agreement only counts
    # when both lists pass: the pair competes as one bloc, then splits its seats.
    passing = votes[votes >= threshold * total_valid]
    bloc_of = {party: party for party in passing.index}
    for first, second in agreements:
        if first in bloc_of and second in bloc_of:
            bloc_of[second] = first
    blocs = passing.groupby(bloc_of)
    bloc_seats = dhondt(blocs.sum(), seats)
    allocation = pd.Series(0, index=votes.index)
    for bloc, members in blocs:
        allocation[members.index] = dhondt(members, bloc_seats[bloc])
    return allocation

party_seats = estimate_seats(
    total_party_votes, total_valid_votes, agreements=surplus_agreements
)

plt.ioff()
fig_total, total_ax = plt.subplots(figsize=(10, max(6, len(party_percent) * 0.35)))
plt.ion()
plt.axvline(3.25, ls='--', c='k')
party_percent.plot(
    kind="barh",
    ax=total_ax,
    color=[party_colors.get(party, party_fallback_color) for party in party_percent.index],
)
total_ax.set_title(
    f"סה״כ מצביעים: {total_ballots_cast:,} | קולות כשרים: {total_valid_votes:,}"
)
total_ax.set_xlabel("אחוז מכלל הקולות הכשרים")
total_ax.set_ylabel("מפלגה")
total_ax.xaxis.set_major_formatter(PercentFormatter())
total_ax.grid(axis="x", alpha=0.25)
fig_total.tight_layout()
fig_total.canvas.layout.width = "56%"

party_legend_rows = "".join(
    f"<tr>"
    f"<td style='padding:3px 8px'><span style='display:inline-block;width:12px;height:12px;background:{party_colors.get(party, party_fallback_color)}'></span></td>"
    f"<th style='padding:3px 8px;text-align:right'>{party}</th>"
    f"<td style='padding:3px 8px;text-align:right'>{party_names.get(party, party)}</td>"
    f"<td dir='ltr' style='padding:3px 8px;text-align:right'>{party_percent[party]:.2f}%</td>"
    f"<td style='padding:3px 8px;text-align:right'>{party_seats[party] or '—'}</td>"
    f"</tr>"
    for party in reversed(party_percent.index)
)
party_legend_header = (
    "<tr><th></th><th></th><th></th><th></th>"
    "<th style='padding:3px 8px;text-align:right'>מנדטים (הערכה)</th></tr>"
)
party_legend = HTML(
    value=(
        "<div dir='rtl'><h3>אותיות המפלגות</h3>"
        f"<table style='border-collapse:collapse'>{party_legend_header}{party_legend_rows}</table></div>"
    ),
    layout=Layout(width="22%", max_height="650px", overflow="auto"),
)

# Group by locality code: results and polling-place tables may spell names differently.
city_counts = located_results.assign(
    uncounted=located_results["מצביעים"] == 0
).groupby("סמל ישוב").agg(
    name=("שם ישוב", "first"),
    uncounted=("uncounted", "sum"),
    total=("uncounted", "size"),
)
top_uncounted_cities = (
    city_counts[city_counts["uncounted"] > 0]
    .sort_values(["uncounted", "total"], ascending=False)
    .head(20)
)
uncounted_rows = "".join(
    f"<tr>"
    f"<th style='padding:3px 8px;text-align:right'>{row.name}</th>"
    f"<td dir='ltr' style='padding:3px 8px;text-align:right'>{row.uncounted:,} / {row.total:,}</td>"
    f"</tr>"
    for row in top_uncounted_cities.itertuples()
)
uncounted_table = HTML(
    value=(
        "<div dir='rtl'><h3>"
        f"<span style='display:inline-block;width:12px;height:12px;background:{uncounted_color}'></span> "
        "ישובים עם הכי הרבה קלפיות שטרם נספרו</h3>"
        f"<table style='border-collapse:collapse'>{uncounted_rows}</table></div>"
    ),
    layout=Layout(width="22%", max_height="650px", overflow="auto"),
)
results_summary = HBox(
    [fig_total.canvas, party_legend, uncounted_table],
    layout=Layout(width="90vw", align_items="flex-start"),
)
results_summary


## Export a static page

Writes `2026/results_map.html`: a standalone Leaflet map with the same polling-place regions and a JavaScript hover/click panel, followed by the totals chart (static SVG), the party legend and the uncounted-localities table. It needs no kernel to view.

In [9]:
static_page_template = r"""<!doctype html>
<html lang="he">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Knesset 26 results</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  body { margin: 0; font-family: system-ui, sans-serif; background: #fff; color: #111; }
  header { padding: 12px 16px; direction: rtl; }
  header h1 { margin: 0 0 4px; font-size: 1.5rem; }
  .row { display: flex; flex-wrap: wrap; gap: 16px; padding: 0 16px 16px; align-items: flex-start; }
  #map { flex: 3 1 480px; height: 760px; }
  #panel { flex: 2 1 320px; direction: rtl; max-height: 760px; overflow: auto; }
  #panel h2 { margin: 0 0 6px; }
  .stat { font-size: 1.15rem; margin: 4px 0; }
  .bar-row { display: grid; grid-template-columns: 10em 1fr 4.5em; gap: 6px; align-items: center; margin: 3px 0; }
  .bar-track { background: #eee; height: 14px; }
  .bar { height: 14px; }
  #chart { flex: 3 1 480px; }
  #chart svg { width: 100%; height: auto; }
  .side-table { flex: 1 1 260px; max-height: 650px; overflow: auto; }
  @media (max-width: 700px) { #map { height: 60vh; } }
</style>
</head>
<body>
<header><h1>תוצאות הבחירות לכנסת ה־26</h1><div>__TOTALS__</div></header>
<div class="row">
  <div id="map"></div>
  <div id="panel"><div id="summary"><h2>Hover to display quick summary</h2></div><div id="bars"></div></div>
</div>
<div class="row">
  <div id="chart">__CHART__</div>
  <div class="side-table">__LEGEND__</div>
  <div class="side-table">__UNCOUNTED__</div>
</div>
<script>
const parties = __PARTIES__;
const regions = __REGIONS__;
const boundaries = __BOUNDARIES__;

const map = L.map("map", {preferCanvas: true, minZoom: 6, maxZoom: 15}).setView([31.5, 34.9], 7);
L.tileLayer("https://tile.openstreetmap.org/{z}/{x}/{y}.png", {
  maxZoom: 19, attribution: "&copy; OpenStreetMap contributors",
}).addTo(map);
L.control.scale({imperial: false, position: "bottomleft"}).addTo(map);

function escapeHtml(text) {
  const element = document.createElement("div");
  element.textContent = text;
  return element.innerHTML;
}
function percent(part, whole) {
  return whole ? (part * 100 / whole).toFixed(2) : "0.00";
}

function showSummary(p) {
  const total = p.votes.reduce((sum, votes) => sum + votes, 0);
  let html = "<h2>" + escapeHtml(p.name) + " — קלפי " + escapeHtml(p.number)
    + (p.counted ? "" : " (טרם נספרה)") + "</h2>";
  if (p.address) html += "<div><small>" + escapeHtml(p.address) + "</small></div>";
  const dominant = parties.findIndex(party => party.id === p.dominant_party);
  if (dominant >= 0) {
    html += '<div class="stat"><b>מפלגה דומיננטית: ' + escapeHtml(parties[dominant].name) + "</b></div>";
  }
  html += '<div class="stat">מצביעים: ' + p.voted.toLocaleString() + "</div>";
  html += '<div class="stat">כשרים: ' + p.valid.toLocaleString() + ", " + percent(p.valid, p.voted) + "%</div>";
  html += '<div class="stat">פסולים: ' + p.invalid.toLocaleString() + ", " + percent(p.invalid, p.voted) + "%</div>";
  if (dominant >= 0) {
    const votes = p.votes[dominant];
    html += '<div class="stat">הצבעות למפלגה הדומיננטית: ' + votes.toLocaleString() + ", " + percent(votes, total) + "%</div>";
  }
  document.getElementById("summary").innerHTML = html;
}

function showBars(p) {
  const total = p.votes.reduce((sum, votes) => sum + votes, 0);
  const bars = document.getElementById("bars");
  if (!total) { bars.innerHTML = ""; return; }
  const top = p.votes.map((votes, index) => [votes, index])
    .filter(([votes]) => votes > 0)
    .sort((a, b) => b[0] - a[0])
    .slice(0, 10);
  bars.innerHTML = "<h3>" + escapeHtml(p.name) + " — קלפי " + escapeHtml(p.number) + "</h3>"
    + top.map(([votes, index]) => {
      const share = votes * 100 / total;
      return '<div class="bar-row"><span>' + escapeHtml(parties[index].name) + "</span>"
        + '<div class="bar-track"><div class="bar" style="width:' + share + "%;background:" + parties[index].color + '"></div></div>'
        + '<span dir="ltr">' + share.toFixed(1) + "%</span></div>";
    }).join("");
}

const baseStyle = {color: "black", weight: 1, opacity: 1, dashArray: "9", fillOpacity: 0.5};
const regionLayer = L.geoJSON(regions, {
  style: feature => Object.assign({}, baseStyle, {fillColor: feature.properties.fill}),
  onEachFeature: (feature, layer) => {
    layer.on("mouseover", () => {
      layer.setStyle({color: "white", dashArray: "0", fillOpacity: 0.9});
      showSummary(feature.properties);
    });
    layer.on("mouseout", () => regionLayer.resetStyle(layer));
    layer.on("click", () => { showSummary(feature.properties); showBars(feature.properties); });
  },
}).addTo(map);
L.geoJSON(boundaries, {
  style: {color: "#000000", weight: 2, opacity: 1, fill: false}, interactive: false,
}).addTo(map);
</script>
</body>
</html>
"""

def compact_geometry(geometry):
    # Round to ~1 m to keep the page small. Pointwise rounding skips GEOS's
    # topology pass, which fails on slightly invalid clipped cells; the page
    # only draws the shapes, so validity doesn't matter here.
    return mapping(set_precision(shape(geometry), 1e-5, mode="pointwise"))

def region_fill(properties):
    if not properties["counted"]:
        return uncounted_color
    return party_colors.get(properties["dominant_party"], party_fallback_color)

summary_keys = [
    "name", "address", "number", "voted", "invalid", "valid",
    "dominant_party", "counted",
]
export_regions = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": compact_geometry(feature["geometry"]),
            "properties": {
                **{key: feature["properties"][key] for key in summary_keys},
                "votes": [feature["properties"]["votes"][party] for party in party_columns],
                "fill": region_fill(feature["properties"]),
            },
        }
        for feature in polling_place_regions["features"]
    ],
}
export_boundaries = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": compact_geometry(feature["geometry"]), "properties": {}}
        for feature in locality_boundaries["features"]
    ],
}
export_parties = [
    {
        "id": party,
        "name": party_names.get(party, party),
        "color": party_colors.get(party, party_fallback_color),
    }
    for party in party_columns
]

chart_svg = StringIO()
fig_total.savefig(chart_svg, format="svg", bbox_inches="tight")
chart_svg = chart_svg.getvalue()
chart_svg = chart_svg[chart_svg.index("<svg"):]

def to_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")

uncounted_total = int((located_results["מצביעים"] == 0).sum())
updated_at = pd.Timestamp.now(tz="Asia/Jerusalem").strftime("%d/%m/%Y %H:%M")
page_values = {
    "__TOTALS__": (
        f"סה״כ מצביעים: {total_ballots_cast:,} | קולות כשרים: {total_valid_votes:,}"
        f" | קלפיות שטרם נספרו: {uncounted_total:,} | עודכן: {updated_at}"
    ),
    "__CHART__": chart_svg,
    "__LEGEND__": party_legend.value,
    "__UNCOUNTED__": uncounted_table.value,
    "__PARTIES__": to_json(export_parties),
    "__REGIONS__": to_json(export_regions),
    "__BOUNDARIES__": to_json(export_boundaries),
}
static_page = static_page_template
for token, value in page_values.items():
    static_page = static_page.replace(token, value)

static_page_path = Path("2026/results_map.html")
static_page_path.write_text(static_page, encoding="utf-8")
print(f"Wrote {static_page_path} ({static_page_path.stat().st_size / 1e6:.1f} MB)")

Wrote 2026/results_map.html (10.1 MB)
